# Prompt Engineering: Templates, Few-Shot, Structured Outputs, Tool Use, Evaluation

Prompts are programs written in natural language. Here we treat them like code: templated, validated,
and **evaluated**. A deterministic mock model stands in for any LLM so the notebook runs offline and
stays vendor-neutral. Swap `mock_model` for a real client (any provider) to try it live.

In [ ]:
import sys
from pathlib import Path
for p in [Path.cwd() / 'code', Path('06-generative-ai/prompt-engineering/code')]:
    if p.exists():
        sys.path.insert(0, str(p.resolve()))
import prompt_tools
print('prompt_tools loaded')

## 1. Templates, zero-shot and few-shot
A template with validated variables prevents silent prompt bugs (missing/extra fields).

In [ ]:
from prompt_tools import *
t = PromptTemplate('Summarise the text in {n} words:\n{text}')
print(t.render(n=10, text='LLMs predict tokens.'))
try:
    t.render(text='x')
except KeyError as e:
    print('caught:', e)
print()
print(build_few_shot('Classify sentiment as pos or neg.', [('great film', 'pos'), ('dull plot', 'neg')], 'loved it'))

## 2. Structured outputs with a validate-and-repair loop
Never trust free-form model text for downstream code: extract, validate against a schema, and feed errors back.

In [ ]:
schema = {'type': 'object', 'required': ['label', 'score'],
          'properties': {'label': {'type': 'string', 'enum': ['pos', 'neg']},
                         'score': {'type': 'number', 'minimum': 0, 'maximum': 1}}}
def mock_structured(prompt):
    if 'previous reply was invalid' in prompt:
        return '```json\n{"label": "pos", "score": 0.93}\n```'
    return 'The sentiment is positive, I would say 93%.'   # first try ignores the format
obj, attempts, errs = structured_call(mock_structured, 'Classify: loved it', schema)
print(obj, 'after', attempts, 'attempt(s)')
assert obj['label'] == 'pos' and attempts == 2

## 3. Reasoning prompts without exposing private reasoning
Ask for a **short, user-facing justification and the final answer in separate fields**, and verify answers with
checks or tools. Do not ask a model to dump hidden reasoning; it is not guaranteed to be faithful and many
systems keep it internal. Here the schema carries only `answer` and `justification` (a brief rationale).

In [ ]:
qa_schema = {'type': 'object', 'required': ['answer', 'justification'],
             'properties': {'answer': {'type': 'number'}, 'justification': {'type': 'string'}}}
def mock_math(prompt):
    return '{"answer": 42, "justification": "6 groups of 7 items gives 42."}'
obj, n, errs = structured_call(mock_math, 'There are 6 boxes with 7 items each. How many items?', qa_schema)
print(obj)
assert obj['answer'] == 6 * 7      # programmatic verification beats trusting the explanation

## 4. Tool-use prompting
The prompt declares tools and a strict reply format. The **application** validates and executes calls, never the model.

In [ ]:
tools = [{'name': 'get_weather', 'description': 'Current weather for a city',
          'parameters': {'type': 'object', 'required': ['city'], 'properties': {'city': {'type': 'string'}}}}]
print(render_tool_prompt(tools, 'Is it raining in Oslo?')[:260], '...')
for reply in ['{"tool": "get_weather", "arguments": {"city": "Oslo"}}',
              '{"tool": "delete_files", "arguments": {}}',
              '{"tool": "get_weather", "arguments": {}}']:
    print(parse_tool_call(reply, tools))

## 5. Prompt evaluation
Compare variants on the same labelled cases and report uncertainty (Wilson 95% interval). With tiny test sets, differences are often noise.

In [ ]:
def mock_model(prompt):
    # a toy 'model' that only gets negatives right when it has seen a negative example
    text = prompt.split('Input:')[-1].lower()
    if 'terrible' in text or 'awful' in text:
        return 'neg' if 'Output: neg' in prompt else 'pos'
    return 'pos'
cases = [('terrible service', 'neg'), ('awful food', 'neg'), ('lovely stay', 'pos'), ('great value', 'pos')]
variants = {
    'zero-shot': lambda x: build_few_shot('Classify sentiment as pos or neg.', [], x),
    'few-shot':  lambda x: build_few_shot('Classify sentiment as pos or neg.', [('bad', 'neg'), ('nice', 'pos')], x),
}
for name, r in evaluate_prompt_variants(variants, cases, mock_model, exact_match).items():
    lo, hi = r['ci95']
    print(f"{name:10s} pass={r['pass_rate']:.2f}  95% CI=({lo:.2f}, {hi:.2f})  n={r['n']}")